### Data Ingestion

In [1]:
###Document Structure

from langchain_core.documents import Document

In [ ]:
doc = Document(
    page_content="This is the content of the document.",
    metadata={
        "source": "example.txt",
        "author": "Ayush Awasthi",
        "created_at": "2024-06-01"
    }
)
doc

In [2]:
import os

os.makedirs("../data/text_files",exist_ok=True)

In [ ]:
sample_texts = {
    "../data/text_files/python_intro.txt": """Python is a high-level, interpreted programming language known for its simplicity and readability.
    It was created by Guido van Rossum and first released in 1991. 
    Python's design philosophy emphasizes code readability and a syntax that allows programmers to express concepts in fewer lines of code compared to other programming languages.""",
    "../data/text_files/data_science.txt": """Data science is an interdisciplinary field that uses scientific methods, processes, algorithms and systems to extract knowledge and insights from structured and unstructured data.
    It combines aspects of statistics, computer science, and domain expertise to analyze and interpret complex data sets. 
    Data scientists use various tools and techniques, including machine learning, data mining, and big data analytics, to uncover patterns and make informed decisions based on data.""",
    "../data/text_files/machine_learning.txt": """Machine learning is a subset of artificial intelligence that focuses on the development of algorithms that allow computers to learn from and make predictions or decisions based on data. 
    It involves training models on large datasets to recognize patterns and make informed decisions without being explicitly programmed for specific tasks.
    Common machine learning algorithms include linear regression, decision trees, support vector machines, and neural networks.""",
}

for filepath,content in sample_texts.items():
    with open(filepath, "w") as f:
        f.write(content)

print("Sample text files created successfully.")

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
documents = loader.load()

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls = TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False
)

documents = dir_loader.load()
documents

In [4]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)
documents = dir_loader.load()
documents

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

print(len(chunks))

96


In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self ,texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    def get_embedding_dimension(self) -> int:
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()
    
embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding Model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16727.20it/s]


Model Loaded Successfully. Embedding dimension: 384


/tmp/ipykernel_78178/3244368085.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [7]:
import os

class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector Store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection. {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_document(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"

            ids.append(doc_id)

            metadata = dict(doc.metadata)

            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)

            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(
                embedding.tolist()
            )

        # OUTSIDE LOOP
        try:

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} documents"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:

            print(f"Error adding documents: {e}")
            raise

vector_store = VectorStore()
vector_store

Vector Store initialized. Collection: pdf_documents
Existing documents in collection. 96


In [8]:
texts = [doc.page_content for doc in chunks]


embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_document(chunks, embeddings)

Generating embeddings for 96 texts...


Batches: 100%|██████████| 3/3 [00:00<00:00, 10.29it/s]

Generated embeddings with shape: (96, 384)
Adding 96 documents to vector store...
Successfully added 96 documents
Total documents in collection: 192


In [9]:
class RagRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int=5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: {query}")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")

        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(query_embeddings=[query_embeddings.tolist()], n_results=top_k)

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i,(doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1-distance

                    if (similarity_score >= score_threshold):
                        retrieved_docs.append({
                            'id':doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RagRetriever(vector_store, embedding_manager)

In [10]:
rag_retriever.retrieve("Backpropagation computes gradients across layers")

Retrieving documents for query: Backpropagation computes gradients across layers
Top k: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.82it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_90f60455_60',
  'content': "Backpropagation computes gradients across layers using the chain rule. Optimizers such as Adam, RMSProp, and\nSGD are responsible for parameter updates.\nConvolutional Neural Networks excel at spatial learning tasks, while Transformers dominate sequence modeling and\nlanguage tasks.\nimport tensorflow as tf\nfrom tensorflow.keras.models import Sequential\nfrom tensorflow.keras.layers import Dense\nmodel = Sequential([\n    Dense(128, activation='relu'),\n    Dense(64, activation='relu'),",
  'metadata': {'creator': '(unspecified)',
   'title': '(anonymous)',
   'file_path': '../data/pdf/deep_learning_guide.pdf',
   'moddate': '2026-05-12T05:04:04+00:00',
   'creationdate': '2026-05-12T05:04:04+00:00',
   'format': 'PDF 1.4',
   'producer': 'ReportLab PDF Library - www.reportlab.com',
   'doc_index': 60,
   'modDate': "D:20260512050404+00'00'",
   'subject': '(unspecified)',
   'content_length': 483,
   'keywords': '',
   'total_pages': 7,
   'tr

In [11]:
from google import genai
from dotenv import load_dotenv

load_dotenv()

# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

response = client.models.generate_content(
    model="gemini-3-flash-preview", contents="Explain how AI works in a few words"
)
print(response.text)


AI finds patterns in data to make predictions.


In [12]:
class GeminiLLM:
    def __init__(self, model_name="gemini-3-flash-preview"):
        self.model_name = model_name
        self.client = genai.Client(
            api_key=os.environ["GEMINI_API_KEY"]
        )

    def generate_response(self, query: str, context: str):
        
        prompt = f"""
You are a knowledgeable AI assistant.

Use the provided context as the primary foundation for your answer.

You may reinforce and expand upon the context using your own knowledge,
but only when the additional information is directly relevant
and semantically connected to the provided context.

Do not introduce unrelated topics or unsupported claims.

Provide clear, detailed, and educational explanations.

Context:
{context}

Question:
{query}

Answer:
"""

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt
        )

        return response.text
    
gemini_llm = GeminiLLM()

In [13]:
query = "What is machine learning"

retrieved_docs = rag_retriever.retrieve(
    query=query,
    top_k=3,
    score_threshold=0.2
)

combined_context = "\n\n".join(
    doc["content"] for doc in retrieved_docs
)

response = gemini_llm.generate_response(
    query=query,
    context=combined_context
)

print(response)

Retrieving documents for query: What is machine learning
Top k: 3, Score threshold: 0.2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 183.94it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


Machine learning (ML) is a specialized branch of **Artificial Intelligence (AI)** focused on building systems that can learn from data, identify patterns, and make decisions with minimal human intervention. 

Instead of being explicitly programmed with specific rules to perform a task, machine learning algorithms use statistical techniques to "train" on datasets, allowing them to improve their performance over time as they are exposed to more information.

### Core Concepts of Machine Learning

To understand machine learning, it is helpful to look at how it differs from traditional computing and how it functions:

1.  **Traditional Programming vs. Machine Learning:** 
    *   In **traditional programming**, a human developer writes specific rules (code) and provides data to produce an output. 
    *   In **machine learning**, a human provides the data and the desired output, and the computer creates the model (the rules) that connects the two.
2.  **Algorithms and Models:** An **algori

In [14]:
class AdvancedRAGPipeline:

    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm

    def query(
        self,
        query: str,
        top_k: int = 5,
        min_score: float = 0.2,
        return_context: bool = False
    ):

        print(f"Processing Query: {query}")

        # STEP 1 — Retrieve documents
        results = self.retriever.retrieve(
            query=query,
            top_k=top_k,
            score_threshold=min_score
        )

        # STEP 2 — Handle empty retrieval
        if not results:
            return {
                "answer": "No relevant context found.",
                "sources": [],
                "confidence": 0.0,
                "context": ""
            }

        # STEP 3 — Build combined context
        context = "\n\n".join(
            [doc["content"] for doc in results]
        )

        # STEP 4 — Extract sources
        sources = []

        for doc in results:

            sources.append({
                "source": doc["metadata"].get(
                    "source",
                    "unknown"
                ),
                "page": doc["metadata"].get(
                    "page",
                    "unknown"
                ),
                "score": round(
                    doc["similarity_score"],
                    3
                ),
                "preview": doc["content"][:150]
            })

        # STEP 5 — Compute confidence
        confidence = max(
            doc["similarity_score"]
            for doc in results
        )

        # STEP 6 — Build prompt
        prompt = f"""
You are an intelligent AI assistant.

Use the retrieved context as the primary source
for answering the question.

You may reinforce the context using your own knowledge,
but do not introduce information unrelated to the context.

Provide:
- detailed explanations
- examples where useful
- intuitive understanding

Context:
{context}

Question:
{query}

Answer:
"""

        # STEP 7 — Generate response
        response = self.llm.generate_response(
            query=query,
            context=context
        )

        # STEP 8 — Final structured output
        output = {
            "answer": response,
            "sources": sources,
            "confidence": round(confidence, 3)
        }

        # OPTIONAL — Include full context
        if return_context:
            output["context"] = context

        return output

rag_pipeline = AdvancedRAGPipeline(
    retriever=rag_retriever,
    llm=gemini_llm
)

result = rag_pipeline.query(
    query="Machine Learning",
    top_k=5,
    min_score=0.2,
    return_context=True
)

print(result["answer"])

for source in result["sources"]:
    print(source)


Processing Query: Machine Learning
Retrieving documents for query: Machine Learning
Top k: 5, Score threshold: 0.2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 387.61it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant context found.


In [15]:
query = "What are transformers?"

results = rag_retriever.retrieve(
    query=query,
    top_k=5,
    score_threshold=0.0
)

print("\n===== RETRIEVED RESULTS =====\n")

for i, r in enumerate(results):

    print(f"\nRESULT {i+1}")
    print(f"Score: {r['similarity_score']}")

    print("\nCONTENT:")
    print(r["content"][:500])

    print("\nSOURCE:")
    print(r["metadata"])

    print("\n" + "="*80)

Retrieving documents for query: What are transformers?
Top k: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 350.96it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)

===== RETRIEVED RESULTS =====

